# Predict Future Sales — Overview / DQC / ETL

Three parts in one notebook:

1. Overview — load everything and see what we got.
2. DQC — go through the data and write down every issue we find.
3. ETL — read that list and actually clean the data based on it. Output is a couple of files ready for further analysis


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.1f}'.format)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

INPUT = Path('input')
CACHE = Path('cache')
CACHE.mkdir(exist_ok=True)

# 1. Overview

Just load everything and look at what's inside

In [ ]:
# load all the csv files. compact dtypes helps save memory 
sales = pd.read_csv(
    INPUT / 'sales_train.csv',
    dtype={'date_block_num': 'int8', 'shop_id': 'int8', 'item_id': 'int32',
           'item_price': 'float32', 'item_cnt_day': 'float32'},
    parse_dates=['date'], date_format='%d.%m.%Y',
)
items = pd.read_csv(INPUT / 'items.csv', dtype={'item_id': 'int32', 'item_category_id': 'int16'})
categories = pd.read_csv(INPUT / 'item_categories.csv', dtype={'item_category_id': 'int16'})
shops = pd.read_csv(INPUT / 'shops.csv', dtype={'shop_id': 'int8'})
test = pd.read_csv(INPUT / 'test.csv', dtype={'ID': 'int32', 'shop_id': 'int8', 'item_id': 'int32'})
submission = pd.read_csv(INPUT / 'sample_submission.csv')

datasets = {
    'sales_train': sales,
    'items': items,
    'item_categories': categories,
    'shops': shops,
    'test': test,
    'sample_submission': submission,
}
print('loaded', len(datasets), 'tables')

In [ ]:
# for each table print shape, dtypes and a couple of rows
for name, df in datasets.items():
    print(f'\n  {name}  shape={df.shape}, {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
    print('dtypes:')
    print(df.dtypes.to_string())
    print('head:')
    print(df.head().to_string())

In [ ]:
# numeric summary of the main table + a few basic facts
desc = sales.select_dtypes(include='number').describe()

# format: count as integer with thousands separator, rest as floats with 2 decimal places
fmt = desc.copy().astype(object)
for col in desc.columns:
    fmt.loc['count', col] = f"{desc.loc['count', col]:,.0f}"
    for row in ['mean', 'std', 'min', '25%', '50%', '75%', 'max']:
        fmt.loc[row, col] = f"{desc.loc[row, col]:,.2f}"

print('sales_train — numeric summary\n')
print(fmt.to_string())

print()
print(f'date range: {sales["date"].min().date()} → {sales["date"].max().date()}')
print(f'date median: {pd.Timestamp(sales["date"].median()).date()}')
print(f'months: {sales["date_block_num"].nunique()}')
print(f'shops: {sales["shop_id"].nunique():,} of {len(shops):,}')
print(f'items: {sales["item_id"].nunique():,} of {len(items):,}')
print(f'test: {len(test):,} pairs ({test["shop_id"].nunique()} shops × {test["item_id"].nunique():,} items)')

## 2. DQC (Data Quality Check)

Go through the data and collect all issues into a list. We don't fix anything here — just make a record. The ETL section below uses this list.

Stuff we look for:
- missing values, duplicate rows
- outliers in price and quantity
- broken references (id in one table missing from another)
- duplicated shops, non-physical shops mixed in
- items with very short history, cold-start items in test

In [ ]:
# a small helper to collect issues. each check appends a row here
issues = []

def log_issue(dataset, issue, count, action):
    issues.append({
        'dataset': dataset,
        'issue': issue,
        'count': int(count),
        'action': action,
    })

In [ ]:
# check 1 - missing values and full duplicate rows
print('missing values:')
for name, df in datasets.items():
    na = int(df.isna().sum().sum())
    print(' ', name, 'NaN =', na)
    if na > 0:
        log_issue(name, 'missing values', na, 'dropna in ETL')

print('\nduplicates:')
for name, df in datasets.items():
    d = int(df.duplicated().sum())
    print(' ', name, 'duplicates =', d)
    if d > 0:
        log_issue(name, 'duplicate rows', d, 'drop_duplicates in ETL')

In [ ]:
# check 2 - price outliers. look at tails of the distribution
neg_price = int((sales['item_price'] <= 0).sum())
extreme_price = int((sales['item_price'] > 100_000).sum())

print('item_price <= 0 :', neg_price)
print('item_price > 100k :', extreme_price)

plt.figure(figsize=(11, 4))
plt.hist(sales['item_price'].clip(-100, 5000).to_numpy(), bins=80, color='slategray')
plt.title('item_price distribution (clipped -100..5000)')
plt.xlabel('price (RUB)')
plt.tight_layout()
plt.show()

if neg_price:
    log_issue('sales_train', 'item_price <= 0', neg_price, 'drop in ETL')
if extreme_price:
    log_issue('sales_train', 'item_price > 100k', extreme_price, 'drop in ETL')

In [ ]:
# check 3 - quantity outliers. negative = returns, big numbers mean bulk or errors
returns = int((sales['item_cnt_day'] < 0).sum())
bulk = int((sales['item_cnt_day'] > 1000).sum())

print('item_cnt_day < 0 :', returns)
print('item_cnt_day > 1000 :', bulk)

plt.figure(figsize=(11, 4))
plt.hist(sales['item_cnt_day'].clip(-5, 20).to_numpy(), bins=40, color='slategray')
plt.title('item_cnt_day distribution (clipped -5..20)')
plt.xlabel('units')
plt.tight_layout()
plt.show()

if returns:
    log_issue('sales_train', 'item_cnt_day < 0 (returns)', returns, 'drop in ETL')
if bulk:
    log_issue('sales_train', 'item_cnt_day > 1000', bulk, 'drop in ETL')

In [ ]:
# check 4 - integrity between tables
checks = [
    #checking that for each sales table column there exist the same column in reference table
    ('sales.item_id not in items', int(sales.loc[~sales['item_id'].isin(items['item_id']), 'item_id'].nunique())),
    ('test.item_id not in items', int(test.loc[~test['item_id'].isin(items['item_id']), 'item_id'].nunique())),
    ('sales.shop_id not in shops', int(sales.loc[~sales['shop_id'].isin(shops['shop_id']), 'shop_id'].nunique())),
    ('items.item_category_id not in item_categories', int(items.loc[~items['item_category_id'].isin(categories['item_category_id']), 'item_category_id'].nunique())),
]
for label, cnt in checks:
    print(' ', label, ':', cnt)
    if cnt > 0:
        log_issue('cross-table', label, cnt, 'reject unmatched rows in ETL')

In [ ]:
# check 5 - shop issues
# (a) some shops are the same physical store with two different ids
# (b) some 'shops' aren't real stores (online shop, mobile trade) 
shop_dup_map = {0: 57, 1: 58, 10: 11, 39: 40}
non_physical = {12: 'online store', 55: 'digital warehouse', 9: 'mobile trade'}

print('duplicated shops (same point, two ids):')
print(shops[shops['shop_id'].isin(list(shop_dup_map) + list(shop_dup_map.values()))].to_string())

print('\nnon-physical shops:')
print(shops[shops['shop_id'].isin(non_physical)].to_string())

log_issue('shops', 'duplicated shops (same point)', len(shop_dup_map),
          f'merge ids in ETL via map {shop_dup_map}')
log_issue('shops', 'non-physical shops in city column', len(non_physical),
          'tag channel=online/offline/mobile in ETL')

In [ ]:
# check 6 - poor dynamics + cold start
# how many items have very short history, and how many test items never appeared in train
item_months = sales.groupby('item_id')['date_block_num'].nunique()
short_3 = int((item_months <= 3).sum())
cold_start = set(test['item_id']) - set(sales['item_id'])

print('items with <= 3 months of history:', short_3, 'of', len(item_months))
print('cold-start items in test (not seen in train):', len(cold_start), 'of', test['item_id'].nunique())

plt.figure(figsize=(11, 4))
plt.hist(item_months.values, bins=34, color='slategray') # type: ignore
plt.title('item lifetime (months with sales)')
plt.xlabel('active months')
plt.ylabel('items')
plt.tight_layout()
plt.show()

log_issue('sales_train', 'items with <= 3 months of history', short_3,
          'flag as short_life in ETL, lag features unreliable')
log_issue('test', 'cold-start items (no history in train)', len(cold_start),
          'fallback to category mean in ETL')

In [ ]:
# save the list of issues to a file. ETL section below will read it.
issues_df = pd.DataFrame(issues).sort_values('count', ascending=False).reset_index(drop=True)
issues_df.to_csv(CACHE / 'dqc_issues.csv', index=False)
print('total issues found:', len(issues_df))
issues_df

# 3. ETL

Now we use the list of issues from DQC and actually clean the data. Output: a cleaned row-level table and a monthly aggregate ready for analysis.

In [ ]:
# step 1 - drop rows flagged in DQC: missing, duplicates, price outliers, qty outliers, returns
before = len(sales)
mask = (
    (sales['item_price'] > 0)
    & (sales['item_price'] < 100_000)
    & (sales['item_cnt_day'] > 0)
    & (sales['item_cnt_day'] < 1000)
)
sales = sales.loc[mask].dropna().drop_duplicates().copy()
print('rows dropped:', before - len(sales), '(', round((before - len(sales)) / before * 100, 2), '%)')

In [ ]:
# step 2 - fix shop issues: merge duplicated ids, tag online/offline channels
shop_dup_map = {0: 57, 1: 58, 10: 11, 39: 40}
non_physical = {12: ('Online', 'online'), 55: ('Online', 'online'), 9: ('Mobile', 'offline')}

sales['shop_id'] = sales['shop_id'].replace(shop_dup_map).astype('int8')
test['shop_id'] = test['shop_id'].replace(shop_dup_map).astype('int8')

shops['clean_name'] = shops['shop_name'].str.replace(r'^!', '', regex=True).str.strip()
shops['city'] = shops['clean_name'].str.split().str[0]
shops['channel'] = 'offline'
for sid, (city, ch) in non_physical.items():
    shops.loc[shops['shop_id'] == sid, ['city', 'channel']] = [city, ch]

print(shops[['shop_id', 'city', 'channel']].head().to_string())

In [ ]:
# step 3 - time-related features and revenue
sales['year'] = sales['date'].dt.year.astype('int16')
sales['month'] = sales['date'].dt.month.astype('int8')
sales['quarter'] = sales['date'].dt.quarter.astype('int8')
sales['dow'] = sales['date'].dt.dayofweek.astype('int8')
sales['dom'] = sales['date'].dt.day.astype('int8')
sales['revenue'] = (sales['item_price'] * sales['item_cnt_day']).astype('float32')

In [ ]:
# step 4 - join dimension tables. also build a rougher category_group from category_name.
categories['category_group'] = categories['item_category_name'].str.split(' - ').str[0].str.strip()

# drop columns that might already exist from a previous run of this cell to keep merges clean
sales = sales.drop(columns=['item_category_id', 'category_group', 'city', 'channel'], errors='ignore')

sales = sales.merge(items[['item_id', 'item_category_id']], on='item_id', how='left')
sales = sales.merge(categories[['item_category_id', 'category_group']], on='item_category_id', how='left')
sales = sales.merge(shops[['shop_id', 'city', 'channel']], on='shop_id', how='left')

print('final sales shape:', sales.shape)
sales.head()

In [ ]:
# step 5 - aggregate to monthly level (shop, item, month). this is the main sense of the task target
monthly = (
    sales
    .groupby(['date_block_num', 'shop_id', 'item_id'], observed=True)
    .agg(
        item_cnt_month=('item_cnt_day', 'sum'),
        revenue_month=('revenue', 'sum'),
        avg_price=('item_price', 'mean'),
        txn_count=('item_cnt_day', 'size'),
    )
    .reset_index()
)
monthly['item_cnt_month_clip'] = monthly['item_cnt_month'].clip(0, 20).astype('float32')

monthly = monthly.merge(items[['item_id', 'item_category_id']], on='item_id', how='left')
monthly = monthly.merge(categories[['item_category_id', 'category_group']], on='item_category_id', how='left')
monthly = monthly.merge(shops[['shop_id', 'city', 'channel']], on='shop_id', how='left')

print('monthly shape:', monthly.shape)

In [ ]:
# step 6 - save the cleaned tables to files so later notebooks can load them quickly
sales.to_parquet(CACHE / 'sales_clean.parquet', index=False)
monthly.to_parquet(CACHE / 'monthly.parquet', index=False)
print('saved sales_clean and monthly')

## Summary


We loaded all the source tables, walked through them to see what kind of data we have (about 3 million transactions across 60 shops and 22k items, covering January 2013 through October 2015). Then we did a DQC: a bunch of checks, and every problem we ran into got logged into a list, which we then saved to a file. After that, the ETL step took that list and actually cleaned up the data : dropped bad rows, fixed shop ids, added some useful columns, and built a monthly aggregate. At the end we have two output files ready to be used for further analysis.

Problems caught along the way:
1) a few extreme price values (one absurdly high, one negative)
2) one row with quantity over 2000 units which looks like a bulk sale or a typo
3) around 7000 rows of returns (negative quantity) that aren't really sales
4) 6 fully duplicated rows
5) 4 pairs of shop ids that turned out to be the same physical store
6) 3 "shops" that aren't real stores at all : online shop and mobile trade ,they used to sit in the city column as garbage
7) about a quarter of all items only show up for 3 months or less, so any lag-based feature on them is going to be unreliable
8) 363 items appear in the test set but were never seen in training — typical problem

Total of about 7k rows got dropped, which is roughly 0.25% of the data , so the dataset is mostly clean.